# TP2 — Exercise 2

Run the cells from top to bottom. This notebook includes its prerequisite classes.

### Exercise 2: Missing value strategies
Each strategy accepts a DataFrame and returns a cleaned copy. Mean filling applies only to numeric columns. Mode filling works for text too; a completely empty column stays missing because it has no mode.

In [1]:
import pandas as pd
from abc import ABC, abstractmethod
from functools import wraps
from time import perf_counter

#### Setup from earlier exercises

In [2]:
class CSVReader:
    def __init__(self, file_path):
        self.file_path = file_path
        self.data = None

    def read(self):
        self.data = pd.read_csv(self.file_path)
        return self.data

    def preview(self, n=5):
        if self.data is None:
            self.read()
        print(self.data.head(n))

In [3]:
reader = CSVReader("Data/sample_data (2).csv")
raw_data = reader.read()

#### Exercise 2 implementation

In [4]:
class MissingValueStrategy(ABC):
    @abstractmethod
    def handle(self, df):
        pass


class DropMissing(MissingValueStrategy):
    def handle(self, df):
        return df.dropna().copy()


class FillMean(MissingValueStrategy):
    def handle(self, df):
        cleaned = df.copy()
        for column in cleaned.select_dtypes(include="number").columns:
            cleaned[column] = cleaned[column].fillna(cleaned[column].mean())
        return cleaned


class FillMode(MissingValueStrategy):
    def handle(self, df):
        cleaned = df.copy()
        for column in cleaned.columns:
            modes = cleaned[column].mode()
            if not modes.empty:
                cleaned[column] = cleaned[column].fillna(modes.iloc[0])
        return cleaned


class DataCleaner:
    def __init__(self, strategy):
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.handle(df)

In [5]:
cleaner = DataCleaner(FillMean())
mean_data = cleaner.clean(raw_data)
print(mean_data)

# The cleaner can switch behavior without changing its clean method.
cleaner.strategy = DropMissing()
print("Rows after dropping missing values:", len(cleaner.clean(raw_data)))

   id     name    age  height_cm  weight_kg         city  score
0   1    Alice  29.00      165.0       68.0     New York   85.0
1   2      Bob  30.25      172.0       70.0  Los Angeles   90.0
2   3  Charlie  35.00      168.0       72.0      Chicago   86.0
3   4    David  30.25      167.0       80.0      Houston   75.0
4   5      Eva  27.00      160.0       55.0     New York   88.0
5   6    Frank  30.00      175.0       85.0  Los Angeles   86.0
6   7    Grace  30.25      162.0       60.0      Chicago   92.0
Rows after dropping missing values: 2
